# Practical 4: Word2Vec Training

**Concept:** In the previous practicals we represented words as sparse vectors (BoW / TF-IDF) that ignore word *meaning*. **Word2Vec** learns dense, low-dimensional **word embeddings** where each word is a vector of real numbers, and words appearing in similar contexts end up with similar vectors.

Word2Vec is based on the **distributional hypothesis**: *"You shall know a word by the company it keeps."* It offers two architectures:
- **CBOW** (`sg=0`): predicts the target word from its surrounding context words.
- **Skip-gram** (`sg=1`): predicts the surrounding context words from the target word.

In this notebook we train a Word2Vec model with **gensim**, then explore the learned embeddings: similar words, word similarities, odd-one-out and analogies.

## Install Gensim
`gensim` is the standard Python library for training word embedding models like Word2Vec.

In [ ]:
!pip install gensim

## Import Required Libraries

In [ ]:
from gensim.models import Word2Vec

import numpy as np
import matplotlib.pyplot as plt

# For PCA-based visualization of the embeddings
from sklearn.decomposition import PCA

## Create the Corpus
Word2Vec expects the corpus as a list of sentences, where each sentence is a list of word tokens.
We use a small corpus grouped into **topic clusters** (AI/ML, NLP, fruits, sports, space, sentiment) so the model can learn which words belong together.


In [ ]:
# Small corpus grouped by topic
sentences = [
    ["machine", "learning", "trains", "models", "on", "data"],
    ["neural", "networks", "are", "deep", "learning", "models"],
    ["deep", "learning", "is", "a", "subset", "of", "machine", "learning"],
    ["a", "model", "learns", "patterns", "from", "training", "data"],
    ["artificial", "intelligence", "powers", "modern", "systems"],
    ["machine", "learning", "models", "learn", "from", "data"],
    ["neural", "networks", "are", "powerful", "for", "classification"],
    ["deep", "learning", "models", "learn", "from", "training", "data"],
    ["artificial", "intelligence", "is", "used", "in", "healthcare"],
    ["natural", "language", "processing", "analyzes", "text"],
    ["word", "embeddings", "represent", "the", "meaning", "of", "words"],
    ["nlp", "tasks", "include", "translation", "and", "summarization"],
    ["a", "token", "is", "a", "single", "word", "in", "a", "text"],
    ["chatbots", "use", "natural", "language", "processing"],
    ["nlp", "systems", "perform", "entity", "recognition"],
    ["word", "embeddings", "capture", "word", "meaning"],
    ["language", "models", "understand", "human", "text"],
    ["embeddings", "are", "used", "as", "features", "for", "nlp", "tasks"],
    ["word2vec", "learns", "a", "dense", "vector", "for", "every", "word"],
    ["apple", "mango", "banana", "and", "grape", "are", "fruits"],
    ["an", "apple", "is", "a", "sweet", "fruit"],
    ["i", "ate", "a", "ripe", "mango", "and", "a", "banana"],
    ["apple", "and", "mango", "are", "tropical", "fruits"],
    ["she", "picked", "a", "fresh", "banana", "from", "the", "tree"],
    ["the", "basket", "had", "apple", "mango", "and", "grape"],
    ["banana", "and", "apple", "are", "common", "fruits"],
    ["a", "ripe", "mango", "tastes", "sweet"],
    ["grape", "and", "apple", "make", "good", "juice"],
    ["baseball", "players", "hit", "the", "ball", "with", "a", "bat"],
    ["the", "team", "won", "the", "championship", "game"],
    ["football", "and", "baseball", "are", "team", "sports"],
    ["the", "players", "practiced", "hard", "before", "the", "game"],
    ["a", "home", "run", "wins", "the", "baseball", "game"],
    ["the", "stadium", "cheered", "for", "the", "winning", "team"],
    ["cricket", "and", "baseball", "use", "a", "bat", "and", "ball"],
    ["the", "rocket", "was", "launched", "into", "orbit"],
    ["astronauts", "travel", "to", "the", "space", "station"],
    ["the", "rocket", "carried", "cargo", "into", "orbit"],
    ["a", "satellite", "orbits", "the", "earth", "in", "space"],
    ["the", "spaceship", "flew", "past", "the", "moon"],
    ["rockets", "launch", "satellites", "into", "earth", "orbit"],
    ["astronauts", "explore", "space", "from", "the", "station"],
    ["the", "movie", "was", "amazing", "and", "heartwarming"],
    ["the", "film", "was", "boring", "and", "terrible"],
    ["reviewers", "called", "the", "story", "exciting", "and", "beautiful"],
    ["the", "performance", "was", "praised", "and", "loved"],
    ["critics", "hated", "the", "dull", "and", "slow", "movie"],
    ["the", "ending", "was", "surprising", "and", "wonderful"],
    ["audiences", "enjoyed", "the", "funny", "and", "clever", "show"],
]

print("Training Corpus")
print("-" * 50)
print(f"Total sentences: {len(sentences)}")
print(f"Total words: {sum(len(s) for s in sentences)}")
print()
for s in sentences:
    print(s)


## Train the Word2Vec Model
We train a **CBOW** model (`sg=0`) with 20-dimensional vectors.
- `vector_size=20` → each word becomes a 20-D vector
- `window=3` → 3 words of context on each side
- `min_count=1` → keep even words that appear once
- `epochs=500` → more passes let the model learn better patterns on a tiny corpus
- `sg=0` → CBOW architecture


In [ ]:
# Train the Word2Vec model (CBOW)
model = Word2Vec(
    sentences,
    vector_size=20,
    window=3,
    min_count=1,
    workers=4,
    sg=0,
    epochs=500,
    seed=42
)

print("Word2Vec Model Trained Successfully!")


## Model Vocabulary
The trained model's vocabulary is every unique word in the corpus.

In [ ]:
# List the entire vocabulary
vocab = model.wv.index_to_key

print("Vocabulary")
print("-" * 50)
print(vocab)
print(f"\nVocabulary size: {len(vocab)}")

## Word Vector
Each word is now a dense 20-dimensional vector of real numbers. Let's look at the vector for the word *"machine"*.


In [ ]:
# Get the embedding vector for a word
vector = model.wv["machine"]

print("Vector for 'machine'")
print("-" * 50)
print(vector)
print(f"\nVector dimension: {len(vector)}")

## Most Similar Words
`most_similar` returns the words whose vectors are closest (highest cosine similarity) to a given word. These are the words that appear in similar contexts.

In [ ]:
# Find the most similar words to 'machine'
similar_words = model.wv.most_similar("machine", topn=10)

print("Most similar words to 'machine'")
print("-" * 50)
for word, score in similar_words:
    print(f"{word:<15} {score:.4f}")

## Word Similarity
We can directly measure the cosine similarity between the vectors of any two words. Related words should score higher.

In [ ]:
# Cosine similarity between two word vectors
sim_machine_learning = model.wv.similarity("machine", "learning")
sim_machine_healthcare = model.wv.similarity("machine", "healthcare")
sim_apple_mango = model.wv.similarity("apple", "mango")
sim_apple_rocket = model.wv.similarity("apple", "rocket")

print("Word Similarities")
print("-" * 50)
print(f"similarity('machine', 'learning')   = {sim_machine_learning:.4f}")
print(f"similarity('machine', 'healthcare') = {sim_machine_healthcare:.4f}")
print(f"similarity('apple', 'mango')       = {sim_apple_mango:.4f}")
print(f"similarity('apple', 'rocket')      = {sim_apple_rocket:.4f}")


## Odd One Out
`doesnt_match` finds the word in a list whose vector is least similar to the rest.

In [ ]:
# Which word doesn't belong?
odd_one = model.wv.doesnt_match(["machine", "learning", "neural", "apple"])

print("Odd One Out")
print("-" * 50)
print(f"Odd one out from [machine, learning, neural, apple] -> {odd_one}")


## Word Analogies
Embeddings support **analogy solving** through vector arithmetic: `word2vec - word + embeddings ≈ vector`.
We probe this with `most_similar(positive=[...], negative=[...])`.


In [ ]:
# Analogy: word2vec - word + embeddings = ?
analogy = model.wv.most_similar(
    positive=["word2vec", "embeddings"],
    negative=["word"]
)

print("Analogy: word2vec - word + embeddings = ?")
print("-" * 50)
for word, score in analogy[:5]:
    print(f"{word:<15} {score:.4f}")


## CBOW vs Skip-gram
For comparison, we also train a **Skip-gram** model (`sg=1`). Skip-gram tends to work better on small datasets and for rare words, while CBOW is faster.

In [ ]:
# Train a Skip-gram model for comparison
skipgram_model = Word2Vec(
    sentences,
    vector_size=20,
    window=3,
    min_count=1,
    workers=4,
    sg=1,
    epochs=500,
    seed=42
)

print("CBOW nearest to 'machine':", [w for w, _ in model.wv.most_similar("machine", topn=3)])
print("Skip-gram nearest to 'machine':", [w for w, _ in skipgram_model.wv.most_similar("machine", topn=3)])


## Visualizing Embeddings (PCA)
The 20-D vectors are hard to inspect, so we reduce them to 2-D with **PCA** and plot the words. Words with similar meaning should cluster together.


In [ ]:
# Words we want to plot
plot_words = [
    "machine", "learning", "deep", "neural",
    "networks", "word2vec", "embeddings", "vector",
    "text", "language", "data", "models"
]

# Gather the vectors
vectors = np.array([model.wv[w] for w in plot_words])

# Reduce to 2 dimensions
pca = PCA(n_components=2)
reduced = pca.fit_transform(vectors)

# Plot
plt.figure(figsize=(10, 6))
plt.scatter(reduced[:, 0], reduced[:, 1], color="steelblue")

for i, word in enumerate(plot_words):
    plt.annotate(word, (reduced[i, 0], reduced[i, 1]), fontsize=12)

plt.title("Word2Vec Embeddings Visualized with PCA")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.grid(alpha=0.3)
plt.show()

## Save the Model
Trained models can be saved to disk and reloaded later — no need to retrain.

In [ ]:
# Save the trained model
model.save("word2vec_model.model")

print("Model saved as word2vec_model.model")

## Load the Model
Loading the saved model restores the exact same vocabulary and vectors.

In [ ]:
# Load the model back from disk
loaded_model = Word2Vec.load("word2vec_model.model")

print("Model loaded successfully!")
print("-" * 50)
print("Vocabulary size:", len(loaded_model.wv.index_to_key))
print("Most similar to 'learning':", loaded_model.wv.most_similar("learning", topn=3))

## Summary

In this practical we:
1. Trained a **Word2Vec CBOW** model using gensim on a small topic-clustered corpus.
2. Explored the learned 20-D embeddings via `most_similar`, `similarity`, `doesnt_match` and analogies.
3. Compared **CBOW** vs **Skip-gram** architectures.
4. Visualized the embeddings with PCA.
5. Saved and reloaded the model.

These dense embeddings can now be used as features for downstream NLP tasks such as sentiment analysis, text classification and NER — covered in the upcoming practicals.
